# Rare Disease Variant Annotation Pipeline — Google Colab Runner

This notebook runs the `diseasesvariants_annotation` pipeline directly inside
**Google Colab** — no need to set up your own machine.

**What this notebook does:**
1. Clones the repository
2. Installs all required tools (bcftools, VEP, SnpEff, SpliceAI, etc.)
3. Downloads the reference databases (GRCh38, ClinVar, ClinGen)
4. Builds the demo lookup tables (gnomAD/dbNSFP/InterVar subsets)
5. Runs the pipeline on the demo VCF (4 example rare-disease variants)
6. Displays the final annotated VCF results
7. Explains how to use **your own disease/VCF file** instead (Section 11)

> **Time & Disk Warning:** The first full setup run can take **30–60 minutes**
> (due to the VEP cache + ClinVar download), and will use ~15–20 GB of disk
> space. Colab's free-tier disk is usually ~70–100 GB, so this should not be
> an issue.

> **Runtime:** Under `Runtime > Change runtime type`, CPU is sufficient — a
> GPU is not required.


## 1. Environment Check

In [ ]:
!lsb_release -a
!df -h /content
!nproc


## 2. Clone the Repository


In [ ]:
%cd /content
!git clone https://github.com/Eiesha-Asif/diseasesvariants_annotation.git repo
%cd /content/repo
!ls -la


## 3. Install All Tools

This step runs `scripts/setup_tools.sh` as-is (no pipeline file is modified).
It installs: `bcftools`, `samtools`, `bedtools`, **Ensembl VEP** (+ GRCh38
cache), **SnpEff/SnpSift**, **AnnotSV**, **ClassifyCNV**, **ISV**, and
**SpliceAI** (in its own conda environment).

**Be patient — this step takes a while.**


In [ ]:
%%bash
cd /content/repo
bash scripts/setup_tools.sh 2>&1 | tail -200


## 4. PATH / Environment Setup (Colab-Specific)

Colab's non-interactive shells do not automatically source `~/.bashrc`, so we
create a small `colab_env.sh` file here and `source` it at the start of every
following cell — this ensures `vep`, `AnnotSV`, `ClassifyCNV`, and `conda`
are all found on the PATH.


In [ ]:
%%bash
cat > /content/repo/colab_env.sh << 'EOF'
export PATH="/content/repo/tools/ensembl-vep:/content/repo/tools/AnnotSV/bin:/content/repo/tools/ClassifyCNV:$PATH"
if [ -d "$HOME/miniconda3" ]; then
  export PATH="$HOME/miniconda3/bin:$PATH"
fi
EOF
cat /content/repo/colab_env.sh

source /content/repo/colab_env.sh
echo "---- verifying tools ----"
vep --help 2>&1 | head -3 || echo "vep not found yet"
bcftools --version | head -1


## 5. Download the Reference Databases

This step runs `scripts/setup_databases.sh`: it downloads the GRCh38
reference genome, the NCBI ClinVar VCF, and the ClinGen dosage sensitivity
BED file.


In [ ]:
%%bash
source /content/repo/colab_env.sh
cd /content/repo
bash scripts/setup_databases.sh


## 6. Build the Demo Lookup Tables (gnomAD / dbNSFP / InterVar Subsets)

`scripts/databases_source/*.tsv` already contains the gnomAD frequency,
dbNSFP scores, and InterVar classification for the 4 demo diseases (Apert,
FOP, Hereditary Hemochromatosis, AATD). Below, those tsv files are copied
(stripping comment/header lines) to their proper `databases/<source>/*.bed`
locations, then sorted, bgzipped, and tabix-indexed.


In [ ]:
%%bash
source /content/repo/colab_env.sh
cd /content/repo

mkdir -p databases/gnomad databases/myvariant databases/intervar

# Strip comment lines (#...) and keep only tab-separated data
grep -v '^#' scripts/databases_source/gnomad_subset.tsv \
  | grep -E '^chr' > databases/gnomad/gnomad_subset.bed

grep -v '^#' scripts/databases_source/dbnsfp_subset.tsv \
  | grep -E '^chr' > databases/myvariant/dbnsfp_subset.bed

grep -E '^chr[0-9XYM]+[[:space:]]' scripts/databases_source/intervar_subset.tsv \
  > databases/intervar/intervar_subset.bed

echo "---- preview ----"
head -3 databases/gnomad/gnomad_subset.bed
head -3 databases/myvariant/dbnsfp_subset.bed
head -3 databases/intervar/intervar_subset.bed

bash scripts/build_example_databases.sh databases refs

echo "---- indexed files ----"
ls -lh databases/gnomad/ databases/myvariant/ databases/intervar/


## 7. Create the Config File (With Colab Paths)

This creates a **new** config file, `config/annotation_resources_colab.env`
— the original `config/annotation_resources.env` file is left untouched.


In [ ]:
%%bash
mkdir -p /content/repo/config
cat > /content/repo/config/annotation_resources_colab.env << 'EOF'
# -------------------- core reference --------------------
REF_FASTA="/content/repo/refs/Homo_sapiens.GRCh38.dna.primary_assembly.fa"
JAVA_MEM="4g"

# -------------------- VEP --------------------
VEP_CACHE_DIR="/content/repo/refs/vep_cache"
RUN_VEP=1

# -------------------- SnpEff --------------------
RUN_SNPEFF=1
SNPEFF_JAR="/content/repo/tools/snpEff/snpEff.jar"
SNPEFF_GENOME="GRCh38.86"

# -------------------- ClinVar --------------------
RUN_CLINVAR=1
CLINVAR_VCF_GZ="/content/repo/databases/clinvar/clinvar.chr.vcf.gz"

# -------------------- gnomAD --------------------
RUN_GNOMAD=1
GNOMAD_VCF_GZ="/content/repo/databases/gnomad/gnomad_subset.bed.gz"

# -------------------- ClinGen --------------------
RUN_CLINGEN=1
CLINGEN_DOSAGE_BED_GZ="/content/repo/databases/clingen/clingen_dosage.hg38.bed.gz"

# -------------------- SpliceAI --------------------
RUN_SPLICEAI=1
SPLICEAI_ASSEMBLY="grch38"

# -------------------- REVEL/AlphaMissense/CADD --------------------
RUN_DBNSFP_BED=1
DBNSFP_BED_GZ="/content/repo/databases/myvariant/dbnsfp_subset.bed.gz"

# -------------------- ACMG/AMP classification --------------------
RUN_INTERVAR_BED=1
INTERVAR_BED_GZ="/content/repo/databases/intervar/intervar_subset.bed.gz"

# -------------------- CNV (optional) --------------------
CLASSIFYCNV_DIR="/content/repo/tools/ClassifyCNV"
ISV_DIR="/content/repo/tools/ISV"
ISV_CONDA_ENV="isv_env"
ANNOTSV_INSTALL_DIR="/content/repo/tools/AnnotSV"
EOF
echo "Config file ready:"
cat /content/repo/config/annotation_resources_colab.env


## 8. Run the Pipeline — On the Demo VCF (4 Rare Diseases)

This runs `rare_disease_vcf_annotation_pipeline.sh` **completely unmodified**
— only the new Colab config is passed in.


In [ ]:
%%bash
source /content/repo/colab_env.sh
if [ -d "$HOME/miniconda3" ]; then
  source "$HOME/miniconda3/etc/profile.d/conda.sh"
  conda activate spliceai_env
fi

cd /content/repo
bash rare_disease_vcf_annotation_pipeline.sh \
  -i four_disease_variants_AATD_Apert_HH_FOP_GRCh38.vcf \
  -o results/colab_demo \
  -c config/annotation_resources_colab.env \
  -s colab_demo_sample \
  -a GRCh38 \
  -t 2


## 9. View the Final Annotated Output

In [ ]:
%%bash
cd /content/repo
echo "---- output files ----"
ls -lh results/colab_demo/snv/

echo
echo "---- first 3 annotated variant lines ----"
zcat results/colab_demo/snv/colab_demo_sample.final.small_variants.annotated.vcf.gz | grep -v "^##" | head -4

echo
echo "---- summary report ----"
cat results/colab_demo/reports/colab_demo_sample.annotation_outputs.txt


## 10. View the Output as a Table (Python)

Below, the annotated VCF is loaded into a `pandas` DataFrame and displayed as
an easy-to-read table — showing each variant's gene, consequence, ClinVar
significance, and gnomAD frequency.


In [ ]:
import gzip
import pandas as pd
import re

vcf_path = "/content/repo/results/colab_demo/snv/colab_demo_sample.final.small_variants.annotated.vcf.gz"

rows = []
with gzip.open(vcf_path, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        chrom, pos, vid, ref, alt, qual, flt, info = cols[:8]
        info_dict = {}
        for kv in info.split(";"):
            if "=" in kv:
                k, v = kv.split("=", 1)
                info_dict[k] = v
        rows.append({
            "CHROM": chrom,
            "POS": pos,
            "REF": ref,
            "ALT": alt,
            "CLNSIG": info_dict.get("CLNSIG", ""),
            "CLNDN": info_dict.get("CLNDN", ""),
            "GNOMAD_AF": info_dict.get("GNOMAD_AF", ""),
            "INTERVAR_GENE": info_dict.get("INTERVAR_GENE", ""),
            "INTERVAR_ACMG": info_dict.get("INTERVAR_ACMG", ""),
        })

df = pd.DataFrame(rows)
df


## 11. Running This on Your Own Disease / Your Own VCF File

This section moves beyond the demo to run the pipeline on **your own
disease's** variants.

### 11.1 — Upload Your VCF File


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload your own .vcf file here
my_vcf = list(uploaded.keys())[0]
print("Uploaded file:", my_vcf)


### 11.2 — Add gnomAD / dbNSFP / InterVar Rows

Retrieve the gnomAD frequency, dbNSFP score, and InterVar classification for
your variant(s) (commands are in README.md Section 7), then add a row to
each of the following files:

```
scripts/databases_source/gnomad_subset.tsv
scripts/databases_source/dbnsfp_subset.tsv
scripts/databases_source/intervar_subset.tsv
```

Then re-run the Section 6 `build_example_databases.sh` step below so the new
rows get indexed too.


In [ ]:
%%bash
source /content/repo/colab_env.sh
cd /content/repo

grep -v '^#' scripts/databases_source/gnomad_subset.tsv | grep -E '^chr' > databases/gnomad/gnomad_subset.bed
grep -v '^#' scripts/databases_source/dbnsfp_subset.tsv | grep -E '^chr' > databases/myvariant/dbnsfp_subset.bed
grep -E '^chr[0-9XYM]+[[:space:]]' scripts/databases_source/intervar_subset.tsv > databases/intervar/intervar_subset.bed

rm -f databases/gnomad/gnomad_subset.bed.gz* databases/myvariant/dbnsfp_subset.bed.gz* databases/intervar/intervar_subset.bed.gz*
bash scripts/build_example_databases.sh databases refs
echo "Lookup tables rebuilt."


### 11.3 — Run the Pipeline on Your VCF

In [ ]:
%%bash
source /content/repo/colab_env.sh
if [ -d "$HOME/miniconda3" ]; then
  source "$HOME/miniconda3/etc/profile.d/conda.sh"
  conda activate spliceai_env
fi

cd /content/repo
MY_VCF="REPLACE_WITH_YOUR_UPLOADED_FILENAME.vcf"   # <-- put the filename from the upload cell above here
MY_SAMPLE="my_disease_sample"

bash rare_disease_vcf_annotation_pipeline.sh \
  -i "$MY_VCF" \
  -o "results/${MY_SAMPLE}" \
  -c config/annotation_resources_colab.env \
  -s "$MY_SAMPLE" \
  -a GRCh38 \
  -t 2


### 11.4 — View Your Final Output


In [ ]:
%%bash
cd /content/repo
MY_SAMPLE="my_disease_sample"
zcat "results/${MY_SAMPLE}/snv/${MY_SAMPLE}.final.small_variants.annotated.vcf.gz" | grep -v "^##" | head -10


## 12. Download Results to Your Local Machine


In [ ]:
from google.colab import files
# set your sample name here (use 'colab_demo_sample' for the demo run)
sample = "colab_demo_sample"
files.download(f"/content/repo/results/colab_demo/snv/{sample}.final.small_variants.annotated.vcf.gz")


## 13. Notes

- This notebook does **not** modify any original pipeline files
  (`rare_disease_vcf_annotation_pipeline.sh`, `scripts/*.sh`) — it only
  creates a new Colab-specific config file
  (`config/annotation_resources_colab.env`).
- The CNV steps (AnnotSV/ClassifyCNV/ISV) are optional — you can enable them
  by passing a `-n <cnv_file>` flag (see the "CNV / Structural Variant"
  section in README.md).
- This pipeline is intended for research/educational use, not as a clinical
  diagnostic tool — always have final classifications verified by a
  qualified geneticist.
- See `README.md` and `LAB_MANUAL.md` for further detail.
